# Notebook ამზადებს `getdata.raw` სქემაში არსებულ მონაცემებს დეშბორდისთვის

`getdata.raw.MotogpTest` ცხრილიდან დამუშავებული შედეგი ინახება `getdata.calculated.MotogpTest` ცხრილში.

---

#### შესრულებული ოპერაციები და ლოგიკა

1. **საკვანძო ველების შენარჩუნება:**
   სკრიპტი ინარჩუნებს ტელემეტრიის 8 ძირითად პარამეტრს: `SpeedKmH`, `CornerRadiusM`, `TrackGrip`, `BikeMassKg`, `TrackTemperature`, `TrackBankingDeg`, `TireType` და `LeanAngleDeg`.

2. **ფიზიკის პარამეტრების გამოთვლა (Feature Engineering):**

   * **`TheoreticalLeanDeg`**: ითვლის მოტოციკლის თეორიულ გადახრის კუთხეს ფიზიკის ფორმულით **tan(θ) = v² / (g × r)** (იდეალურ ფიზიკურ პირობებში). სიჩქარე გადაჰყავს **მ/წმ**-ში (`SpeedKmH / 3.6`), ითვლის არკტანგენსს (`ATAN`) და რადიანებს გარდაქმნის გრადუსებში (`DEGREES`), რათა შეედაროს რეალურ `LeanAngleDeg`-ს, მაგრამ რადგან იდეალური ფიზიკური პირობები ხანდახან არარეალურია საჭიროა ზედა ზღვრის გათვალისწინება, რომელიც რეალურ პირობებში 68° გადახრის კუთხეა.
   * **`CentripetalAccelMs2`**: ითვლის ცენტრისკენულ აჩქარებას **a_c = v² / r**, რაც გვიჩვენებს სავალ ნაწილზე და საბურავებზე მოქმედ **g-force** დატვირთვას.

3. **მონაცემთა კატეგორიზაცია:**

   * **`CornerType`**: ჯგუფავს მოსახვევებს რადიუსის მიხედვით (`მკვეთრი <80m`, `საშუალო 80–150m`, `ფართო >150m`).
   * **`TemperatureZone`**: ჯგუფავს ტრასის ტემპერატურას (`გრილი <30°C`, `ოპტიმალური 30–40°C`, `ცხელი >40°C`).
   * **`TrackInclination`**: ჯგუფავს Banking-ის კუთხეს (`დაღმართი <0°`, `ბრტყელი 0–4°`, `აღმართი >4°`).
   * **`GripLevel`**: ჯგუფავს ტრასის friction ინდექსს (`დაბალი <0.85`, `საშუალო 0.85–0.95`, `მაღალი >0.95`).

---


In [0]:
%sql
CREATE OR REPLACE TABLE getdata.calculated.MotogpTest AS
SELECT 
    SpeedKmH,
    CornerRadiusM,
    TrackGrip,
    BikeMassKg,
    TrackTemperature,
    TrackBankingDeg,
    TireType,
    LeanAngleDeg,
    
    CAST(
        LEAST(
            DEGREES(
                ATAN(
                    POWER(SpeedKmH / 3.6, 2) / (CornerRadiusM * 9.81)
                )
            ),
            68.0
        ) AS DECIMAL(10,2)
    ) AS TheoreticalLeanDeg,

    CAST(
        POWER(SpeedKmH / 3.6, 2) / CornerRadiusM AS DECIMAL(10,2)
    ) AS CentripetalAccelMs2,
    
    CASE 
        WHEN CornerRadiusM < 80 THEN 'მკვეთრი (<80m)'
        WHEN CornerRadiusM <= 150 THEN 'საშუალო (80-150m)'
        ELSE 'ფართო (>150m)'
    END AS CornerType,
    
    CASE 
        WHEN TrackTemperature < 30 THEN 'გრილი (<30°C)'
        WHEN TrackTemperature <= 40 THEN 'ოპტიმალური (30-40°C)'
        ELSE 'ცხელი (>40°C)'
    END AS TemperatureZone,
    
    CASE 
        WHEN TrackBankingDeg < 0 THEN 'დაღმართი (<0°)'
        WHEN TrackBankingDeg <= 4 THEN 'ბრტყელი (0-4°)'
        ELSE 'აღმართი (>4°)'
    END AS TrackInclination,

    CASE 
        WHEN TrackGrip < 0.85 THEN 'დაბალი (<0.85)'
        WHEN TrackGrip <= 0.95 THEN 'საშუალო (0.85-0.95)'
        ELSE 'მაღალი (>0.95)'
    END AS GripLevel

FROM getdata.raw.MotogpTest;

## მონაცემების შემოწმება

In [0]:
%sql
SELECT *
FROM getdata.calculated.MotogpTest